In [ ]:
"""
Search for LPG points of sale (filling + reseller) in all capitals and second cities
of continental sub-Saharan Africa (including Madagascar).
Sources: OpenStreetMap (Overpass) + Google Places (multilingual).
Output: single CSV, JSON, GeoPackage for all cities, plus a summary CSV with statistics.

Keywords are commented out for cost control.
"""

import requests
import time
import csv
import json
import os
import math
import random
from datetime import datetime
from collections import Counter

# ======================== CONFIGURATION ========================
USE_OSM = True                     # ## CHANGE ## flag to enable/disable OSM 
USE_GOOGLE = False
API_KEY = "LA_TUA_API_KEY_GOOGLE"  # <-- insert valid key

RADIUS = 50000                     # radius in metres for OSM & Google Nearby Search

# Output root folder
OUTPUT_ROOT = "output_africa_onlyFEW_CITIES_onlyosm2"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Timestamp for final output files
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Global list that holds all points from all cities (used for intermediate saves)
global_all_results = []


# ======================== KEYWORDS ========================
# ----- RESELLER (reseller) -----
RESELLER_KEYWORDS = {
    'en': [
        # "LPG refill station", "gas station with LPG", "LPG filling point",
        # "butane gas", "butane cylinder", "LPG dealer", "gas cylinder exchange",
        "cooking gas", "LPG"
    ],
    'fr': [
        # "station de remplissage de GPL", "station-service avec GPL",
        # "bouteille de gaz butane", "point de remplissage de GPL",
        # "revendeur de GPL", "échange de bouteilles de gaz",
        "gaz de cuisine", "GPL", "gaz butane"
    ],
    'pt': [
        # "estação de recarga de GLP", "posto de gasolina com GLP",
        # "gás butano", "botija de gás butano", "ponto de enchimento de GLP",
        # "revendedor de GLP", "troca de botijão de gás",
        "gás de cozinha", "GLP"
    ],
    'sw': [
        # "kituo cha kujaza LPG", "kituo cha mafuta chenye LPG",
        # "sehemu ya kujaza LPG", "muuzaji wa LPG",
        # "kubadilisha mitungi ya gesi", "gesi ya butane", "mtungi wa gesi ya butane",
        "gesi ya kupikia", "LPG"
    ],
    'ar': [
        # "محطة تعبئة غاز البترول المسال", "محطة وقود مع غاز البترول المسال",
        # "غاز البيوتان", "اسطوانة غاز البيوتان", "نقطة تعبئة غاز البترول المسال",
        # "تاجر غاز البترول المسال", "استبدال اسطوانات الغاز",
        "غاز الطبخ", "غاز البترول المسال"
    ]
}

# ----- FILLING PLANTS (filling) -----
FILLING_KEYWORDS = {
    'en': [
        "LPG cylinder filling plant", "LPG bottling plant",
        # "gas bottling plant", "LPG filling plant",
        "cylinder filling station", "LPG bottling station"
    ],
    'fr': [
        "usine de remplissage de bouteilles de GPL", "usine d'embouteillage de GPL",
        # "usine d'embouteillage de gaz", "usine de remplissage de GPL",
        "station de remplissage de bouteilles", "station d'embouteillage de GPL"
    ],
    'pt': [
        "fábrica de enchimento de botijas de GLP", "fábrica de engarrafamento de GLP",
        # "fábrica de engarrafamento de gás", "fábrica de enchimento de GLP",
        "estação de enchimento de botijas", "estação de engarrafamento de GLP"
    ],
    'sw': [
        "kiwanda cha kujaza mitungi ya LPG",
        "kiwanda cha kujaza LPG",
        # "kiwanda cha kujaza gesi", "kiwanda cha kusindika LPG",
        "kituo cha kujaza mitungi", "kituo cha kujaza LPG"
    ],
    'ar': [
        "مصنع تعبئة اسطوانات غاز البترول المسال", "مصنع تعبئة غاز البترول المسال",
        # "مصنع تعبئة الغاز", "محطة تعبئة اسطوانات الغاز",
        "محطة تعبئة غاز البترول المسال", "محطة تعبئة الغاز"
    ]
}

# ======================== LANGUAGE MAP PER COUNTRY ========================
COUNTRY_LANGUAGES = {
    'angola':             ['en', 'pt'],
    'benin':              ['en', 'fr'],
    'botswana':           ['en'],
    'burkina_faso':       ['en', 'fr'],
    'burundi':            ['en', 'fr', 'sw'],
    'cameroon':           ['en', 'fr'],
    'central_african_republic': ['en', 'fr'],
    'chad':               ['en', 'fr', 'ar'],
    'congo':              ['en', 'fr'],
    'congo_drc':          ['en', 'fr', 'sw'],
    'cote_divoire':       ['en', 'fr'],
    'djibouti':           ['en', 'fr', 'ar'],
    'equatorial_guinea':  ['en', 'pt', 'fr'],
    'eritrea':            ['en', 'ar'],
    'eswatini':           ['en'],
    'ethiopia':           ['en'],
    'gabon':              ['en', 'fr'],
    'gambia':             ['en'],
    'ghana':              ['en'],
    'guinea':             ['en', 'fr'],
    'guinea_bissau':      ['en', 'pt'],
    'kenya':              ['en', 'sw'],
    'lesotho':            ['en'],
    'liberia':            ['en'],
    'madagascar':         ['en', 'fr'],
    'malawi':             ['en'],
    'mali':               ['en', 'fr'],
    'mauritania':         ['en', 'fr', 'ar'],
    'mozambique':         ['en', 'pt'],
    'namibia':            ['en'],
    'niger':              ['en', 'fr'],
    'nigeria':            ['en'],
    'rwanda':             ['en', 'fr', 'sw'],
    'senegal':            ['en', 'fr'],
    'sierra_leone':       ['en'],
    'somalia':            ['en', 'ar'],
    'south_africa':       ['en'],
    'south_sudan':        ['en', 'ar', 'sw'],
    'sudan':              ['en', 'ar'],
    'tanzania':           ['en', 'sw'],
    'togo':               ['en', 'fr'],
    'uganda':             ['en', 'sw'],
    'zambia':             ['en'],
    'zimbabwe':           ['en']
}

# ======================== CITY LIST ========================
# Central coordinates for capital and second city; the script performs a single 50 km circle search.
CITIES_TO_PROCESS = [
    # # Angola
      {'city': 'luanda',      'country': 'angola',        'lat': -8.838, 'lon': 13.234},
    # {'city': 'huambo',      'country': 'angola',        'lat':-12.766, 'lon': 15.739},
    # # Benin
    # {'city': 'porto_novo',  'country': 'benin',         'lat': 6.483,  'lon': 2.616},
    # {'city': 'cotonou',     'country': 'benin',         'lat': 6.370,  'lon': 2.428},
    # # Botswana
    # {'city': 'gaborone',    'country': 'botswana',      'lat':-24.628, 'lon': 25.923},
    # {'city': 'francistown', 'country': 'botswana',      'lat':-21.171, 'lon': 27.514},
    # # Burkina Faso
    # {'city': 'ouagadougou', 'country': 'burkina_faso',  'lat': 12.371, 'lon': -1.528},
    # {'city': 'bobo_dioulasso','country': 'burkina_faso','lat': 11.177, 'lon': -4.298},
    # # Burundi
    # {'city': 'gitega',      'country': 'burundi',       'lat': -3.427, 'lon': 29.925},
    # {'city': 'bujumbura',   'country': 'burundi',       'lat': -3.382, 'lon': 29.360},
    # # Cameroon
    # {'city': 'yaounde',     'country': 'cameroon',      'lat': 3.848,  'lon': 11.496},
    # {'city': 'douala',      'country': 'cameroon',      'lat': 4.051,  'lon': 9.768},
    # # Central African Republic
    # {'city': 'bangui',      'country': 'central_african_republic','lat': 4.362, 'lon': 18.583},
    # {'city': 'bimbo',       'country': 'central_african_republic','lat': 4.332, 'lon': 18.554},
    # # Chad
    # {'city': 'n_djamena',   'country': 'chad',          'lat': 12.113, 'lon': 15.049},
    # {'city': 'moundou',     'country': 'chad',          'lat': 8.567,  'lon': 16.083},
    # # Congo
    # {'city': 'brazzaville', 'country': 'congo',         'lat': -4.267, 'lon': 15.283},
    # {'city': 'pointe_noire','country': 'congo',         'lat': -4.776, 'lon': 11.864},
    # # Congo DRC
    # {'city': 'kinshasa',    'country': 'congo_drc',     'lat': -4.326, 'lon': 15.309},
    # {'city': 'lubumbashi',  'country': 'congo_drc',     'lat':-11.688, 'lon': 27.484},
    # # Côte d'Ivoire
    # {'city': 'yamoussoukro','country': 'cote_divoire',  'lat': 6.828,  'lon': -5.277},
    # {'city': 'abidjan',     'country': 'cote_divoire',  'lat': 5.345,  'lon': -4.022},
    # # Djibouti
    # {'city': 'djibouti_city','country': 'djibouti',     'lat': 11.589, 'lon': 43.148},
    # {'city': 'ali_sabieh',  'country': 'djibouti',      'lat': 11.156, 'lon': 42.708},
    # # Equatorial Guinea
    # {'city': 'malabo',      'country': 'equatorial_guinea','lat': 3.750, 'lon': 8.783},
    # {'city': 'bata',        'country': 'equatorial_guinea','lat': 1.867, 'lon': 9.767},
    # # Eritrea
    # {'city': 'asmara',      'country': 'eritrea',       'lat': 15.328, 'lon': 38.932},
    # {'city': 'massawa', 'country': 'eritrea', 'lat': 15.609, 'lon': 39.450},
    # # eSwatini
    # {'city': 'mbabane',     'country': 'eswatini',      'lat':-26.320, 'lon': 31.144},
    # {'city': 'manzini',     'country': 'eswatini',      'lat':-26.503, 'lon': 31.372},
    # # Ethiopia
      {'city': 'addis_ababa', 'country': 'ethiopia',      'lat': 9.024,  'lon': 38.757},
    # {'city': 'dire_dawa',   'country': 'ethiopia',      'lat': 9.593,  'lon': 41.862},
    # # Gabon
    # {'city': 'libreville',  'country': 'gabon',         'lat': 0.390,  'lon': 9.454},
    # {'city': 'port_gentil', 'country': 'gabon',         'lat': -0.719, 'lon': 8.781},
    # # Gambia
    # {'city': 'banjul',      'country': 'gambia',        'lat': 13.453, 'lon':-16.579},
    # {'city': 'serekunda',   'country': 'gambia',        'lat': 13.423, 'lon':-16.678},
    # # Ghana
    # {'city': 'accra',       'country': 'ghana',         'lat': 5.556,  'lon': -0.201},
    # {'city': 'kumasi',      'country': 'ghana',         'lat': 6.688,  'lon': -1.624},
    # # Guinea
    # {'city': 'conakry',     'country': 'guinea',        'lat': 9.509,  'lon':-13.712},
    # {'city': 'nzerekore',   'country': 'guinea',        'lat': 7.758,  'lon': -8.818},
    # # Guinea-Bissau
    # {'city': 'bissau',      'country': 'guinea_bissau', 'lat': 11.860, 'lon':-15.598},
    # {'city': 'bafata',      'country': 'guinea_bissau', 'lat': 12.167, 'lon':-14.663},
    # # Kenya
      {'city': 'nairobi',     'country': 'kenya',         'lat': -1.286, 'lon': 36.817},
    # {'city': 'mombasa',     'country': 'kenya',         'lat': -4.043, 'lon': 39.668},
    # # Lesotho
    # {'city': 'maseru',      'country': 'lesotho',       'lat':-29.316, 'lon': 27.483},
    # {'city': 'teyateyaneng','country': 'lesotho',       'lat':-29.150, 'lon': 27.733},
    # # Liberia
    # {'city': 'monrovia',    'country': 'liberia',       'lat': 6.301,  'lon':-10.797},
    # {'city': 'ganta',       'country': 'liberia',       'lat': 7.234,  'lon': -8.991},
    # # Madagascar
    # {'city': 'antananarivo','country': 'madagascar',    'lat':-18.914, 'lon': 47.536},
    # {'city': 'toamasina',   'country': 'madagascar',    'lat':-18.149, 'lon': 49.402},
    # # Malawi
    # {'city': 'lilongwe',    'country': 'malawi',        'lat':-13.983, 'lon': 33.783},
    # {'city': 'blantyre',    'country': 'malawi',        'lat':-15.786, 'lon': 35.008},
    # # Mali
    # {'city': 'bamako',      'country': 'mali',          'lat': 12.639, 'lon': -8.002},
    # {'city': 'segou',       'country': 'mali',          'lat': 13.431, 'lon': -6.266},
    # # Mauritania
    # {'city': 'nouakchott',  'country': 'mauritania',    'lat': 18.086, 'lon':-15.975},
    # {'city': 'nouadhibou',  'country': 'mauritania',    'lat': 20.933, 'lon':-17.033},
    # # Mozambique
    # {'city': 'maputo',      'country': 'mozambique',    'lat':-25.969, 'lon': 32.573},
    # {'city': 'beira',       'country': 'mozambique',    'lat':-19.840, 'lon': 34.838},
    # # Namibia
    # {'city': 'windhoek',    'country': 'namibia',       'lat':-22.561, 'lon': 17.086},
    # {'city': 'walvis_bay',  'country': 'namibia',       'lat':-22.957, 'lon': 14.505},
    # # Niger
    # {'city': 'niamey',      'country': 'niger',         'lat': 13.512, 'lon': 2.112},
    # {'city': 'zinder',      'country': 'niger',         'lat': 13.807, 'lon': 8.989},
    # # Nigeria
    # {'city': 'abuja',       'country': 'nigeria',       'lat': 9.057,  'lon': 7.489},
      {'city': 'lagos',       'country': 'nigeria',       'lat': 6.455,  'lon': 3.394},
    # # Rwanda
    # {'city': 'kigali',      'country': 'rwanda',        'lat': -1.950, 'lon': 30.060},
    # {'city': 'gisenyi',     'country': 'rwanda',        'lat': -1.702, 'lon': 29.260},
    # # Senegal
    # {'city': 'dakar',       'country': 'senegal',       'lat': 14.694, 'lon':-17.446},
    # {'city': 'touba',       'country': 'senegal',       'lat': 14.861, 'lon':-15.881},
    # # Sierra Leone
    # {'city': 'freetown',    'country': 'sierra_leone',  'lat': 8.484,  'lon':-13.234},
    # {'city': 'bo',          'country': 'sierra_leone',  'lat': 7.962,  'lon':-11.740},
    # # Somalia
    # {'city': 'mogadishu',   'country': 'somalia',       'lat': 2.037,  'lon': 45.343},
    # {'city': 'hargeisa',    'country': 'somalia',       'lat': 9.562,  'lon': 44.077},
    # # South Africa
    # {'city': 'pretoria',    'country': 'south_africa',  'lat':-25.746, 'lon': 28.188},
    # {'city': 'johannesburg','country': 'south_africa',  'lat':-26.204, 'lon': 28.047},
    # # South Sudan
    # {'city': 'juba',        'country': 'south_sudan',   'lat': 4.860,  'lon': 31.591},
    # {'city': 'wau',         'country': 'south_sudan',   'lat': 7.702,  'lon': 27.997},
    # # Sudan
    # {'city': 'khartoum',    'country': 'sudan',         'lat': 15.552, 'lon': 32.560},
    # {'city': 'omdurman',    'country': 'sudan',         'lat': 15.647, 'lon': 32.481},
    # # Tanzania
    # {'city': 'dodoma',      'country': 'tanzania',      'lat': -6.173, 'lon': 35.742},
    # {'city': 'dar_es_salaam','country': 'tanzania',     'lat': -6.792, 'lon': 39.208},
    # # Togo
    # {'city': 'lome',        'country': 'togo',          'lat': 6.128,  'lon': 1.225},
    # {'city': 'sokode',      'country': 'togo',          'lat': 8.991,  'lon': 1.145},
    # # Uganda
      {'city': 'kampala',     'country': 'uganda',        'lat': 0.313,  'lon': 32.581},
    # {'city': 'gulu',        'country': 'uganda',        'lat': 2.772,  'lon': 32.299},
    # # Zambia
    # {'city': 'lusaka',      'country': 'zambia',        'lat':-15.417, 'lon': 28.283},
    # {'city': 'kitwe',       'country': 'zambia',        'lat':-12.823, 'lon': 28.202},
    # # Zimbabwe
      {'city': 'harare',      'country': 'zimbabwe',      'lat':-17.822, 'lon': 31.050},
    # {'city': 'bulawayo',    'country': 'zimbabwe',      'lat':-20.171, 'lon': 28.587}
]

# ======================== OSM SEARCH FUNCTION (single circle) ========================
def osm_search(lat, lon, radius=50000, max_retries=3):
    """
    Performs a single Overpass API query for a circular area around (lat, lon).
    Returns a list of place dictionaries.
    """
    delta = radius / 111000.0
    min_lat = lat - delta
    max_lat = lat + delta
    min_lon = lon - delta
    max_lon = lon + delta

    # Simplified, non‑redundant query
    query = f"""
    [out:json];
    (
        node["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["man_made"="storage_tank"]["content"~"lpg|gpl|glp|gas|butane|propane"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    url = "https://overpass-api.de/api/interpreter"
    headers = {
        'User-Agent': 'LPG-Research-Script/1.0 (contact: your-email@example.com)',
        'Accept': 'application/json'
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, data={'data': query}, headers=headers, timeout=60)
            if response.status_code == 200:
                data = response.json()
                break
            elif response.status_code in [429, 504]:
                wait = (2 ** attempt) + random.uniform(0, 1)   # exponential backoff: 1, 2, 4 seconds
                print(f"  ⚠️ OSM {response.status_code} - retry in {wait:.1f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            else:
                return [], response.status_code
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            else:
                return [], f"Exception({e})"
    else:
        # loop finished without success
        return [], response.status_code

    results = []
    for elem in data.get('elements', []):
        if elem['type'] == 'node':
            lat_place = elem.get('lat')
            lon_place = elem.get('lon')
        else:
            center = elem.get('center', {})
            lat_place = center.get('lat')
            lon_place = center.get('lon')
        if lat_place is None or lon_place is None:
            continue
        tags = elem.get('tags', {})
        if 'amenity' in tags and tags['amenity'] == 'fuel':
            ptype = 'fuel_station'
        elif 'shop' in tags:
            ptype = tags['shop']
        else:
            ptype = 'lpg_related'

        # Determine category
        is_fuel_station = tags.get('amenity') == 'fuel' and (
            tags.get('fuel:lpg') or tags.get('fuel:GPL') or tags.get('fuel:lpg') in ['yes', 'only']
        )
        is_storage_tank = tags.get('man_made') == 'storage_tank' and tags.get('content', '').lower() in ['lpg', 'gpl', 'glp', 'gas', 'butane', 'propane']
        is_industrial_gas = tags.get('industrial') == 'gas'
        category = 'filling' if (is_fuel_station or is_storage_tank or is_industrial_gas) else 'reseller'

        name = tags.get('name', '')
        results.append({
            'place_id': f"osm_{elem['id']}",
            'name': name,
            'address': '',
            'lat': lat_place,
            'lng': lon_place,
            'types': ptype,
            'keyword': 'osm',
            'search_lat': lat,
            'search_lon': lon,
            'rating': None,
            'user_ratings_total': 0,
            'source': 'osm',
            'language': '',
            'category': category
        })
    return results, None

# ======================== GOOGLE PLACES FUNCTIONS ========================
def google_nearby_search(lat, lon, keyword, lang, category):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': RADIUS,
        'keyword': keyword,
        'language': lang,
        'key': API_KEY
    }
    while True:
        resp = requests.get(url, params=params)
        data = resp.json()
        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            error_msg = data.get('error_message', '')
            print(f"  ⚠️ Google status {data['status']} for {keyword}_{lang}: {error_msg}")
            break
        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': f"{keyword}_{lang}",
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0),
                'source': 'google',
                'language': lang,
                'category': category
            })
        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)
        else:
            break
    return results

# ======================== SAVE GLOBAL FILES (intermediate saves) ========================
def save_global_files(results, output_root, timestamp):
    """
    Saves (overwrites) the global CSV, JSON and GeoPackage files.
    This function is called after each city to provide a crash-safe backup.
    """
    if not results:
        return
    base = os.path.join(output_root, f"all_africa_combined_{timestamp}")
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total','source','language','category']

    # CSV
    with open(f"{base}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    # JSON
    with open(f"{base}.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    # GeoPackage (separate layers for filling and reseller)
    try:
        import geopandas as gpd
        import pandas as pd
        from shapely.geometry import Point
        keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                     'rating','user_ratings_total','source','language','category']
        filling = [r for r in results if r['category'] == 'filling']
        reseller = [r for r in results if r['category'] != 'filling']
        if filling:
            df = pd.DataFrame(filling)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg", driver='GPKG', layer='filling')
        if reseller:
            df = pd.DataFrame(reseller)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg", driver='GPKG', layer='reseller')
    except ImportError:
        pass  # GeoPandas not installed

# ======================== PROCESS ONE CITY ========================
def process_city(city_config, country_name, country_langs):
    city_name = city_config['city']
    lat_c, lon_c = city_config['lat'], city_config['lon']

  
    city_results = []
    total_osm_calls = 0
    total_google_calls = 0
    osm_errors = Counter()

    # ---------- OSM ----------
    # ... inside process_city, after calling osm_search(lat_c, lon_c, RADIUS)
    if USE_OSM:
        time.sleep(2)   # pre‑call pause to respect rate limits
        osm_places, err_code = osm_search(lat_c, lon_c, RADIUS, max_retries=5)   # use more retries first
        total_osm_calls += 1

        if not osm_places and err_code is not None:
            # Fallback: 2x2 grid around the city centre (original method)
            print(f"  ↳ Single circle failed (err {err_code}), switching to 4‑point grid.")
            
            # 4 punti: angoli di un quadrato di 0.2° intorno al centro
            offsets = [(-0.1, -0.1), (-0.1, 0.1), (0.1, -0.1), (0.1, 0.1)]
            for dlat, dlon in offsets:
                glat = lat_c + dlat
                glon = lon_c + dlon
                time.sleep(2)
                try:
                    sub_places, sub_err = osm_search(glat, glon, RADIUS, max_retries=5)
                    total_osm_calls += 1
                    if sub_places:
                        osm_places.extend(sub_places)
                except Exception as e:
                    print(f"  ⚠️ Grid point ({glat},{glon}) failed: {e}")

            # Deduplicate
            seen = {}
            for p in osm_places:
                if p['place_id'] not in seen:
                    seen[p['place_id']] = p
            osm_places = list(seen.values())
            
            err_code = None if osm_places else err_code

        if not err_code:
            city_results.extend(osm_places)
        else:
            osm_errors[err_code] += 1

    # ---------- Google ----------
    if USE_GOOGLE and API_KEY != "LA_TUA_API_KEY_GOOGLE":
        # Filling
        for lang in country_langs:
            if lang not in FILLING_KEYWORDS: continue
            for kw in FILLING_KEYWORDS[lang]:
                try:
                    places = google_nearby_search(lat_c, lon_c, kw, lang, 'filling')
                    total_google_calls += 1 + len(places) // 60
                    city_results.extend(places)
                except Exception as e:
                    print(f"  ⚠️ Google filling {kw}_{lang}: {e}")
                time.sleep(0.2)
        # Reseller
        for lang in country_langs:
            if lang not in RESELLER_KEYWORDS: continue
            for kw in RESELLER_KEYWORDS[lang]:
                try:
                    places = google_nearby_search(lat_c, lon_c, kw, lang, 'reseller')
                    total_google_calls += 1 + len(places) // 60
                    city_results.extend(places)
                except Exception as e:
                    print(f"  ⚠️ Google reseller {kw}_{lang}: {e}")
                time.sleep(0.2)


    # Deduplica per place_id (keep the first)
    unique_places_dict = {}   # place_id → record
    for place in city_results:
        pid = place['place_id']
        if pid not in unique_places_dict:
            unique_places_dict[pid] = place
        else:
        # se la nuova occorrenza è filling e quella già salvata no, sostituisci
            if place['category'] == 'filling' and unique_places_dict[pid]['category'] != 'filling':
                unique_places_dict[pid] = place
    city_results = list(unique_places_dict.values())


    # Ricalcola solo sui risultati Google unici
    google_results = [r for r in city_results if r['source'] == 'google']
    lang_counter = Counter(r['language'] for r in google_results) if google_results else Counter()

    

    # Source breakdown
    osm_total = sum(1 for r in city_results if r['source'] == 'osm')
    google_total = len(city_results) - osm_total
    filling = [r for r in city_results if r['category'] == 'filling']
    reseller = [r for r in city_results if r['category'] == 'reseller']


    if osm_errors:
        print(f"  OSM HTTP errors: {dict(osm_errors)}")

    # Rating stats (brief)
    if city_results:
        try:
            ratings = [float(r['rating']) for r in city_results if r.get('rating') is not None]
            if ratings:
                avg_rating = sum(ratings) / len(ratings)
            reviews = [int(r.get('user_ratings_total', 0)) for r in city_results]
            few_reviews = sum(1 for r in reviews if r < 2)
            
        except:
            pass
    # One compact line per city
    print(f"[{city_name.upper()}] {len(city_results)} pts | OSM {osm_total} Google {google_total} | Filling {len(filling)} Reseller {len(reseller)}")


        

    return city_results, lang_counter

# ======================== MAIN ========================
def main():
    global global_all_results
    summary_rows = []   # will hold dicts for summary CSV

    # Group cities by country
    country_cities = {}
    for cfg in CITIES_TO_PROCESS:
        country = cfg['country']
        country_cities.setdefault(country, []).append(cfg)

    for country, city_list in country_cities.items():

        langs = COUNTRY_LANGUAGES.get(country, ['en']) 
        country_results = []

        for cfg in city_list:
            city_results, lang_cnt = process_city(cfg, country, langs)
            country_results.extend(city_results)
            # Add to global list and save intermediate
            global_all_results.extend(city_results)
            save_global_files(global_all_results, OUTPUT_ROOT, TIMESTAMP)   # crash-safe backup
            time.sleep(2)

            # Build summary row for this city
            osm_city = sum(1 for r in city_results if r['source'] == 'osm')
            google_city = len(city_results) - osm_city
            filling_city = sum(1 for r in city_results if r['category'] == 'filling')
            reseller_city = sum(1 for r in city_results if r['category'] == 'reseller')
            # Language breakdown as string
            lang_str = ", ".join(f"{l}:{cnt}({cnt*100//sum(lang_cnt.values())}%)" for l, cnt in lang_cnt.most_common()) if lang_cnt else ""
            summary_rows.append({
                'Città': cfg['city'].upper(),
                'Stato': country.upper(),
                'Totale punti': len(city_results),
                'Totale OSM': osm_city,
                'Totale Google': google_city,
                'Tot Filling': filling_city,
                'Tot Reseller': reseller_city,
                'Lingue Google (%)': lang_str
            })

        # Country statistics
        osm = [r for r in country_results if r['source']=='osm']
        gg = [r for r in country_results if r['source']=='google']
        filling = [r for r in country_results if r['category']=='filling']
        reseller = [r for r in country_results if r['category']=='reseller']
        print(f"[{country.upper()}] {len(country_results)} pts total | OSM {len(osm)} Google {len(gg)} | Filling {len(filling)} Reseller {len(reseller)}")

    # Save final global files (already up‑to‑date from last city, but just in case)
    save_global_files(global_all_results, OUTPUT_ROOT, TIMESTAMP)
    print(f"\n\nGlobal dataset saved with prefix 'all_africa_combined_{TIMESTAMP}'")

    # Save summary CSV
    summary_path = os.path.join(OUTPUT_ROOT, f"summary_{TIMESTAMP}.csv")
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['Città', 'Stato', 'Totale punti', 'Totale OSM', 'Totale Google',
                      'Tot Filling', 'Tot Reseller', 'Lingue Google (%)']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Summary CSV saved: {summary_path}")

    print("Processing completed.")

if __name__ == "__main__":
    if USE_GOOGLE and API_KEY == "LA_TUA_API_KEY_GOOGLE":
        print("⚠️ Insert a valid Google API key in the API_KEY variable.")
    main() 

[LUANDA] 12 pts | OSM 12 Google 0 | Filling 0 Reseller 12
[ANGOLA] 12 pts total | OSM 12 Google 0 | Filling 0 Reseller 12
[ADDIS_ABABA] 5 pts | OSM 5 Google 0 | Filling 4 Reseller 1
[ETHIOPIA] 5 pts total | OSM 5 Google 0 | Filling 4 Reseller 1
  ⚠️ OSM 429 - retry in 1.1s (attempt 1/5)
  ⚠️ OSM 429 - retry in 2.3s (attempt 2/5)
  ⚠️ OSM 429 - retry in 4.9s (attempt 3/5)
[NAIROBI] 16 pts | OSM 16 Google 0 | Filling 12 Reseller 4
[KENYA] 16 pts total | OSM 16 Google 0 | Filling 12 Reseller 4
[LAGOS] 2 pts | OSM 2 Google 0 | Filling 0 Reseller 2
[NIGERIA] 2 pts total | OSM 2 Google 0 | Filling 0 Reseller 2
  ⚠️ OSM 429 - retry in 1.0s (attempt 1/5)
  ⚠️ OSM 429 - retry in 2.3s (attempt 2/5)
  ⚠️ OSM 504 - retry in 4.3s (attempt 3/5)
[KAMPALA] 6 pts | OSM 6 Google 0 | Filling 4 Reseller 2
[UGANDA] 6 pts total | OSM 6 Google 0 | Filling 4 Reseller 2
[HARARE] 2 pts | OSM 2 Google 0 | Filling 0 Reseller 2
[ZIMBABWE] 2 pts total | OSM 2 Google 0 | Filling 0 Reseller 2


Global dataset saved w